# Metrics and synthetic tissue displacement

This notebook uses the active repository model. The synthetic case is the logistic **continuous material flow**, not the discrete chaotic logistic map:

$$r(a,t)=\frac{ae^{0.8t}}{1-a+ae^{0.8t}},\qquad (x,y)=(r,0.15u).$$

Eight orthogonal gene directions generate Poisson counts through Psi=2 sqrt(E). Material labels are hidden during fitting. Stage 2 differentiates the fitted log-E field directly; there is no expression refit or truth-based coordinate calibration.

The default is a small execution check. Set `RUN_FULL=True` for 50 tissues × 100,000 cells, 1,500 epochs, the T25 source, twelve original targets, 2,500 source cells, contour K=316, and RK4 maximum step 0.0025. This takes substantial compute. Reduced runs are not scientific benchmark results.

The earlier tournament used a different Stage 1 and a separately refitted decoder. Its reported improvements cannot be transferred to this model. Direct autodiff RK4 here replaces its velocity-grid interpolation. Install once: `pip install -e ".[notebooks]"`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src/flow_stp").exists():
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from dataclasses import replace
from scipy.stats import spearmanr
from flow_stp.stage1 import Stage1ARunConfig, run_stage1a
from flow_stp.metrics import compute_rpce
from flow_stp.simulations.logistic_displacement import (
    LogisticDisplacementConfig, generate_logistic_tissues,
    coordinates_for_tissues, evaluate_displacement,
)
torch.set_num_threads(2)


In [ ]:
RUN_FULL = False
config = LogisticDisplacementConfig() if RUN_FULL else LogisticDisplacementConfig(
    n_times=7, cells_per_tissue=500, n_source=32, contour_k=8, max_dt=0.05)
OUTPUT = ROOT / "experiments" / ("logistic_full" if RUN_FULL else "logistic_smoke")
data = generate_logistic_tissues(config)
run_config = Stage1ARunConfig(output_dir=OUTPUT / "stage1")
if not RUN_FULL:
    run_config = replace(run_config, epochs=10, batch_size_per_slice=128,
                         diagnostic_cells_per_slice=32, gauge_reference_points=256)
print("FULL BENCHMARK" if RUN_FULL else "SMOKE CHECK ONLY")
result = run_stage1a(**data.stage1_kwargs(), config=run_config)
z_by_tissue = coordinates_for_tissues(result, data)


## Upstream coordinate diagnostics
Signed Spearman correlation is an evaluation diagnostic. Its sign does not calibrate the fitted chart. RPCE compares fractional physical positions along gradient paths; it is optional and should be interpreted alongside anatomical truth and the saved derivative diagnostics.

In [ ]:
coordinate_metrics = pd.DataFrame({
    "time": data.times,
    "signed_spearman_z_x": [spearmanr(z,xy[:,0]).statistic for z,xy in zip(z_by_tissue,data.xy)],
})
coordinate_metrics["absolute_spearman_z_x"] = coordinate_metrics.signed_spearman_z_x.abs()
display(coordinate_metrics)
display(result.coordinate_diagnostics)
coordinate_metrics.to_csv(OUTPUT / "coordinate_truth_metrics.csv", index=False)
result.history.plot(x="epoch", y="data_nll", title="Training Poisson NLL")
plt.show()


In [ ]:
RUN_RPCE = False
if RUN_RPCE:
    rpce = compute_rpce(result.model.coordinate_maps, data.xy,
        xy_norm_params=result.xy_norm_params, n_z=41, n_paths_per_tissue=8)
    display(rpce.summary)
    rpce.per_z.to_csv(OUTPUT / "rpce_by_z.csv", index=False)
    rpce.per_tissue.to_csv(OUTPUT / "rpce_by_tissue.csv", index=False)


## Physical displacement
Predict target z through Fisher projection and RK4. Lift to each target tissue by selecting K nearest z values, then the point nearest the source transverse y. Compare with the known material endpoint.

Primary metric: pooled displacement NRMSE = sqrt(sum squared endpoint errors / sum squared true displacements). Smaller is better; no motion scores 1. Motion skill is 1 − NRMSE². The oracle lift control measures finite target-cell/contour resolution. Tiny/mid/large motion bins use target RMS displacement thresholds 0.01 and 0.03. Both backward and forward targets are evaluated through time reparameterization of the existing RK4 API.

In [ ]:
device = next(result.model.parameters()).device
summary, by_target = evaluate_displacement(data, z_by_tissue,
    result.model.fisher_flat, device=device, output_dir=OUTPUT / "metrics")
display(pd.Series(summary))
display(by_target)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(by_target.time, by_target.pooled_displacement_nrmse, "o-", label="Fitted model")
ax.plot(by_target.time, by_target.oracle_lift_nrmse, "o--", label="Oracle contour lift")
ax.axhline(1, color="gray", linestyle=":", label="No motion")
ax.set(xlabel="Target time", ylabel="Displacement NRMSE")
ax.legend(); fig.tight_layout(); plt.show()
fig.savefig(OUTPUT / "metrics/displacement_nrmse.png", dpi=150)


## Reusing metrics on real tissues
For a Qian `result` and `qian` loaded with the root workflow notebook, call:
```python
rpce = compute_rpce(result.model.coordinate_maps, qian.spatial_by_slice,
                    xy_norm_params=result.xy_norm_params)
```
Real-data H1/layer agreement requires explicit annotations and a stated ordering. Absolute correlation may resolve a single reflection for reporting, but cannot establish physical displacement without an independent correspondence or motion truth. No such truth is invented here.

Outputs include training manifests/checkpoints, coordinate metrics, per-target displacement scores, pooled scores, benchmark configuration, source row indices, and predicted/true endpoints. Generated data and weights belong under `experiments/`, not Git.